# Activation Oracle — GoalScope Coding Context

Tests whether [Karvonen et al. (2025)](https://arxiv.org/pdf/2512.15674)'s pre-trained
**activation oracle** can verbalize a code-LLM's instrumental goal at every token
position in a generated completion.

**Pipeline:**
1. Load model via `ModelandTokenizer`; load oracle LoRA on the underlying HF model
2. Pick a task from `IncompleteCodeDataset` and use the stored generation
3. Collect hidden states at layer 50% for every code token with `collect_hidden_states`
4. For each token: inject its hidden state into the oracle → oracle answers the question
5. Visualize with `show_patchscope` from `src.goals.goalscope`

**What was wrong in the previous version:**
- Used raw `AutoModelForCausalLM` / `AutoTokenizer` instead of `ModelandTokenizer`
- Reimplemented hidden-state extraction with fragile multi-path `_get_layer` heuristics
- Tokenized with `add_special_tokens=True` — double BOS breaks Llama offset mapping
- Reimplemented code-segment finding instead of using `find_token_range(occurrence=1)` from `src.tokens`
- Reimplemented HTML visualization instead of using `show_patchscope`


In [1]:
%load_ext autoreload
%autoreload 2
import sys, os
sys.path.append('..')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

In [2]:
# ── Config ──────────────────────────────────────────────────────────────────
import torch
from pathlib import Path
from src.utils.env_utils import DEFAULT_RESULTS_DIR
from src.activation_oracle import ORACLE_CONFIGS

##############################################################################
MODEL_KEY   = 'meta-llama/Llama-3.3-70B-Instruct'
TASK_NAME   = 'atoi'          # well-known task to probe
LAYER_PCT   = 70              # extraction layer (% of total layers)
MAX_TOKENS  = None            # None = all code tokens; int = quick test
MAX_NEW_TOK = 64              # oracle generation budget per token
ORACLE_Q    = 'What are you doing at this line of the code? Keep your answer as brief as possible.'
##############################################################################

_cfg             = ORACLE_CONFIGS[MODEL_KEY]
ORACLE_LORA_PATH = _cfg['oracle_lora']
MODEL_SHORT      = MODEL_KEY.split('/')[-1]
GEN_DIR          = Path(DEFAULT_RESULTS_DIR) / 'well-known-tasks' / 'generations' / MODEL_SHORT

print(f'Model       : {MODEL_KEY}')
print(f'Oracle LoRA : {ORACLE_LORA_PATH}')
print(f'Layer pct   : {LAYER_PCT}%')
print(f'Task        : {TASK_NAME}')

Model       : meta-llama/Llama-3.3-70B-Instruct
Oracle LoRA : adamkarvonen/checkpoints_act_cls_latentqa_pretrain_mix_adding_Llama-3_3-70B-Instruct
Layer pct   : 70%
Task        : atoi


In [3]:
# ── Load model via ModelandTokenizer ────────────────────────────────────────
from src.models import ModelandTokenizer

mt = ModelandTokenizer(
    model_key=MODEL_KEY,
    dtype=torch.bfloat16,
    device_map='auto',
)
EXTRACT_LAYER = int(mt.n_layer * LAYER_PCT / 100)
print(f'Loaded {MODEL_SHORT} | {mt.n_layer} layers | extract layer = {EXTRACT_LAYER}')

meta-llama/Llama-3.3-70B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Loaded Llama-3.3-70B-Instruct | 80 layers | extract layer = 56


In [4]:
type(mt.model._module)

transformers.models.llama.modeling_llama.LlamaModel

In [5]:
# ── Load oracle LoRA via PeftModel ───────────────────────────────────────────
# mt._model is the raw LlamaForCausalLM (stored directly in nnsight's __dict__,
# not an Envoy).  We pass it to PeftModel.from_pretrained — the LoRA layers are
# injected in-place into mt._model's linears; peft_model is just a thin wrapper
# that adds adapter management (set_adapter / generate / etc.).
#
# We do NOT assign back to mt._model.  nnsight tracing through mt continues to
# work: adapter layers are disabled for target-model runs and LoRA branches
# contribute zero, giving clean base-model activations.

from peft import PeftModel

ORACLE_ADAPTER = ORACLE_LORA_PATH.replace('.', '_')

if not isinstance(mt._model, PeftModel):
    peft_model = PeftModel.from_pretrained(
        mt._model,
        ORACLE_LORA_PATH,
        adapter_name=ORACLE_ADAPTER,
        is_trainable=False,
        low_cpu_mem_usage=True,
    )
    print(f"Oracle LoRA loaded as adapter '{ORACLE_ADAPTER}'")
else:
    peft_model = mt._model
    if ORACLE_ADAPTER not in peft_model.peft_config:
        peft_model.load_adapter(
            ORACLE_LORA_PATH, adapter_name=ORACLE_ADAPTER,
            is_trainable=False, low_cpu_mem_usage=True,
        )
        print(f"Oracle LoRA loaded as adapter '{ORACLE_ADAPTER}'")
    else:
        print('Oracle LoRA already present')

# disable_adapter_layers() is LoraModel's method — does not cascade to the
# transformers PeftAdapterMixin (which would fail with "No adapter loaded").
peft_model.base_model.disable_adapter_layers()

# ── Verify: adapter is disabled (base-model mode for activation collection) ──
n_disabled = sum(
    1 for m in peft_model.modules()
    if hasattr(m, '_disable_adapters') and m._disable_adapters
)
print(f'Adapter modules in disabled state: {n_disabled}  (should be > 0)')
print('Model ready.')

Oracle LoRA loaded as adapter 'adamkarvonen/checkpoints_act_cls_latentqa_pretrain_mix_adding_Llama-3_3-70B-Instruct'
Adapter modules in disabled state: 560  (should be > 0)
Model ready.


In [6]:
# ── Build prompt from stored generation ─────────────────────────────────────
# Use the stored generation so the prompt format exactly matches run_goalscope.
import json as _json

gen_path = GEN_DIR / 'custom' / f'{TASK_NAME}.json'
if not gen_path.exists():
    raise FileNotFoundError(
        f'{gen_path}\n'
        f'Run: python -m scripts.generate_well_known_task_completions '
        f'--model-key="{MODEL_KEY}" --tasks {TASK_NAME}'
    )

gen_data    = _json.loads(gen_path.read_text())
full_prompt = gen_data['prompt'] + gen_data['generation']

print(f"Task          : {gen_data['function_name']}")
print(f"Prompt length : {len(gen_data['prompt'])} chars")
print(f"Generation    : {len(gen_data['generation'])} chars")
print(f"\nCode body (first 400 chars):\n{gen_data['generation'][:400]}")

Task          : atoi
Prompt length : 572 chars
Generation    : 534 chars

Code body (first 400 chars):
Converts a string to an integer."""
    s = s.strip()
    if not s:
        return 0
    sign = -1 if s[0] == '-' else 1
    if s[0] in ['-', '+']:
        s = s[1:]
    num = 0
    for c in s:
        if not c.isdigit():
            break
        num = num * 10 + int(c)
    return max(-2**31, min(sign * num, 2**31 - 1))

def main():
    print(atoi("42")) 
    print(atoi("   -42")) 
    print(atoi


In [7]:
# ── Tokenize + collect hidden states ────────────────────────────────────────
# IMPORTANT: add_special_tokens=False — Llama/Qwen chat templates already
# include BOS; using True would insert a second BOS and corrupt offset indices.
# collect_hidden_states uses nnsight-native mt.trace() so it's Envoy-safe.
# Base weights are already active (disable_adapters called in cell 4).
from src.activation_oracle import collect_hidden_states, find_code_segment

all_hidden, offset_mapping = collect_hidden_states(
    mt, full_prompt, layer=EXTRACT_LAYER
)

# Token ids for find_code_segment length fallback
token_ids = mt.tokenizer(
    full_prompt, add_special_tokens=False
)['input_ids']

print(f'Hidden states : {tuple(all_hidden.shape)}  (seq_len, d_model)')
print(f'Extract layer : {EXTRACT_LAYER}  ({LAYER_PCT}% of {mt.n_layer})')

# find_token_range(occurrence=1) skips the system-prompt example <code> tag
# and finds the second occurrence — the actual assistant code block.
code_start, code_end = find_code_segment(full_prompt, offset_mapping, token_ids)
print(f'Code tokens   : [{code_start} : {code_end}]  = {code_end - code_start} tokens')

if MAX_TOKENS:
    code_end = code_start + MAX_TOKENS
    print(f'Clamped to first {MAX_TOKENS} tokens → [{code_start} : {code_end}]')

Hidden states : (280, 8192)  (seq_len, d_model)
Extract layer : 56  (70% of 80)
Code tokens   : [92 : 280]  = 188 tokens


In [8]:
# ── Smoke test: verify oracle adapter + injection are both working ───────────
# Runs oracle on two tokens with different hidden states and checks that
# (a) the oracle LoRA is actually enabled after set_adapter (PEFT bug check), and
# (b) different activations produce different responses.
import torch as _t

_return_idx = next(
    (i for i in range(code_start, code_end)
     if 'return' in mt.tokenizer.decode([token_ids[i]]).lower()),
    code_start + 10,
)
_ws_idx = code_start  # first token is usually whitespace/newline

print(f"Token A: [{_return_idx}] {repr(mt.tokenizer.decode([token_ids[_return_idx]]))}")
print(f"Token B: [{_ws_idx}]  {repr(mt.tokenizer.decode([token_ids[_ws_idx]]))}")

# Check _disable_adapters AFTER enable+set_adapter (the bug was that set_adapter
# alone did not reset _disable_adapters when adapters were previously disabled)
from src.activation_oracle import query_oracle
peft_model.base_model.enable_adapter_layers()
peft_model.set_adapter(ORACLE_ADAPTER)
n_active = sum(
    1 for m in peft_model.modules()
    if hasattr(m, '_disable_adapters') and not m._disable_adapters
)
peft_model.base_model.disable_adapter_layers()
print(f"\nOracle adapter active modules after enable+set_adapter: {n_active}  (0 = bug still present)")

# Query both tokens
_resp_a = query_oracle(
    peft_model=peft_model, tokenizer=mt.tokenizer,
    activation_vec=all_hidden[_return_idx], question=ORACLE_Q,
    oracle_adapter_name=ORACLE_ADAPTER, extract_layer=EXTRACT_LAYER, max_new_tokens=48,
)
_resp_b = query_oracle(
    peft_model=peft_model, tokenizer=mt.tokenizer,
    activation_vec=all_hidden[_ws_idx], question=ORACLE_Q,
    oracle_adapter_name=ORACLE_ADAPTER, extract_layer=EXTRACT_LAYER, max_new_tokens=48,
)

print(f"\nToken A  → {_resp_a}")
print(f"Token B  → {_resp_b}")
print(f"\n→ {'DIFFERENT ✓  Injection is working.' if _resp_a.strip() != _resp_b.strip() else 'SAME ✗  Responses identical — injection may still be broken.'}")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


Token A: [122] ' return'
Token B: [92]  '\n\n'

Oracle adapter active modules after enable+set_adapter: 560  (0 = bug still present)

Token A  → I am checking if the input is a valid integer.
Token B  → I am checking if the user has entered a valid input.

→ DIFFERENT ✓  Injection is working.


In [ ]:
# ── Per-token oracle queries ─────────────────────────────────────────────────
from src.activation_oracle import query_oracle

results = []   # list[dict] — format for show_patchscope

n_code_toks = code_end - code_start
print(f'Running oracle on {n_code_toks} code tokens ...')
print(f'Question: {ORACLE_Q!r}\n')

for i, tok_idx in enumerate(range(code_start, code_end)):
    tok_str    = mt.tokenizer.decode([token_ids[tok_idx]])
    activation = all_hidden[tok_idx]   # [d_model]

    verbalization = query_oracle(
        peft_model          = peft_model,
        tokenizer           = mt.tokenizer,
        activation_vec      = activation,
        question            = ORACLE_Q,
        oracle_adapter_name = ORACLE_ADAPTER,
        extract_layer       = EXTRACT_LAYER,
        max_new_tokens      = MAX_NEW_TOK,
    )

    results.append({'pos': tok_idx, 'token': tok_str, 'patched_gen': verbalization})

    if i == 0 or (i + 1) % 10 == 0 or i + 1 == n_code_toks:
        print(f'  [{i+1:3d}/{n_code_toks}] {repr(tok_str):<18} -> {verbalization[:70]}')

print('\nDone.')

Running oracle on 188 code tokens ...
Question: 'What are you doing at this line of the code? Keep your answer as brief as possible.'

  [  1/188] '\n\n'             -> I am checking if the user has entered a valid input.
  [ 10/188] ' """'             -> You are checking if the input is a valid integer.
  [ 20/188] ' s'               -> I am checking if the input is a string.
  [ 30/188] '       '          -> I am checking if the input is a valid integer.
  [ 40/188] ' if'              -> I am checking if the variable `a` is less than 0.
  [ 50/188] '\n'               -> You are checking if the variable `x` is less than 10.
  [ 60/188] " '+"              -> I am checking if the input is a valid integer.


In [ ]:
# ── Visualize with show_patchscope ──────────────────────────────────────────
# Uses the project's existing hover-tooltip visualization.
from src.goals.goalscope import show_patchscope

show_patchscope(
    results,
    title=f'Activation Oracle — {TASK_NAME} ({MODEL_SHORT}) — layer {EXTRACT_LAYER}/{mt.n_layer}',
)

In [ ]:
# # ── Save to HTML file (optional) ────────────────────────────────────────────
# save_path = None  # set to e.g. 'results/activation_oracle/atoi.html'

# from src.goals.goalscope import show_patchscope
# show_patchscope(
#     results,
#     title=f'Activation Oracle — {TASK_NAME} ({MODEL_SHORT})',
#     save_path=save_path,
# )

In [ ]:
# # ── [Optional] Compare oracle responses across layers ────────────────────────
# # Probe one token at 25%, 50%, 75% depth to see how representations evolve.

# COMPARE_TOK_IDX = code_start + 10
# tok_str = mt.tokenizer.decode([token_ids[COMPARE_TOK_IDX]])
# print(f'Token: {repr(tok_str)}')

# for pct in [25, 50, 75]:
#     layer = int(mt.n_layer * pct / 100)
#     hs, _ = collect_hidden_states(mt, full_prompt, layer=layer)
#     verb  = query_oracle(
#         peft_model          = peft_model,
#         tokenizer           = mt.tokenizer,
#         activation_vec      = hs[COMPARE_TOK_IDX],
#         question            = ORACLE_Q,
#         oracle_adapter_name = ORACLE_ADAPTER,
#         extract_layer       = layer,
#         max_new_tokens      = MAX_NEW_TOK,
#     )
#     print(f'  Layer {layer:3d} ({pct:2d}%) -> {verb[:80]}')